# Python Testing Script
Use this template to create a Python testing script that imports your CRUD Python module to call and test the create and read instances of CRUD functionality. 
### 1. Import your CRUD Python module
### 2. Instantiate an instance of your class
### 3. Use your create function create a new record in the aac database
### 4. Use your read funtion to return records from the aac database 
### 5. Use your update function to update an existing record in the aac database
### 6. Use your delete function to delete an existing record in the aac database

<div class="alert alert-block alert-success"><span style="color:black;"><b>Tip: </b>You should have created test functionality for items 1 thru 4 during Module Four Milestone, copy over your existing test script code and expand it here to call and test the Update and Delete functionality.</span></div>

In [3]:
# 1. import CRUD module
from CRUD_Python_Module import AnimalShelter

# 2. Instantiate an instance of the class
shelter = AnimalShelter()

# ------------------------------------------------------------------------
# [C]RUD - CREATE
# ------------------------------------------------------------------------
print("---TESTING CREATE---")
test_animal = {
    "name": "Renly",
    "animal_type": "Dog",
    "breed": "Mixed",
    "color": "Brindle",
    "outcome_type": "Adoption"
}

create_success = shelter.create(test_animal)
print(f"Created successful: {create_success}")

# ------------------------------------------------------------------------
# C[R]UD - READ
# ------------------------------------------------------------------------
print("---TESTING READ---")
query_read = {"name": "Renly"}
read_results = shelter.read(query_read)

print(f"C[R]UD- READ returned {len(read_results)} document(s).")
for doc in read_results:
    print(doc)

# ------------------------------------------------------------------------
# CR[U]D - UPDATE
# ------------------------------------------------------------------------
print("---TESTING UPDATE---")
update_filter = {"name": "Renly"}
update_data = {"name": "Baratheon"}

modified_count = shelter.update(update_filter, update_data)
print(f"CR[U]D- UPDATE modified {modified_count} document(s).")

# Verify record update via read
updated_records = shelter.read({"name": "Baratheon"})
print(f"Verification Read returned {len(updated_records)} document(s).")
for doc in updated_records:
    print(doc)

# ------------------------------------------------------------------------
# CRU[D] - DELETE
# ------------------------------------------------------------------------
print("---TESTING DELETE---")
delete_filter = {"name": "Baratheon"}

deleted_count = shelter.delete(delete_filter)
print(f"CRU[D]- DELETE removed {deleted_count} document(s).")

# Verify deletion via read (should return 0 documents) 
post_delete_records = shelter.read(delete_filter)
print(f"Verification Read after delete returned {len(post_delete_records)} document(s).")


---TESTING CREATE---
Created successful: True
---TESTING READ---
C[R]UD- READ returned 1 document(s).
{'_id': ObjectId('6abd3194a59d4a07d5b5f5fe'), 'name': 'Renly', 'animal_type': 'Dog', 'breed': 'Mixed', 'color': 'Brindle', 'outcome_type': 'Adoption'}
---TESTING UPDATE---
CR[U]D- UPDATE modified 1 document(s).
Verification Read returned 1 document(s).
{'_id': ObjectId('6abd3194a59d4a07d5b5f5fe'), 'name': 'Baratheon', 'animal_type': 'Dog', 'breed': 'Mixed', 'color': 'Brindle', 'outcome_type': 'Adoption'}
---TESTING DELETE---
CRU[D]- DELETE removed 1 document(s).
Verification Read after delete returned 0 document(s).


In [8]:
"""Robust testing of Animal Shelter Class"""

import unittest
from CRUD_Python_Module import AnimalShelter


class TestAnimalShelterModule(unittest.TestCase):
   
    @classmethod
    def setUpClass(cls):
        """Instantiate the CRUD class once before running test suite"""
        cls.shelter = AnimalShelter()
        cls.test_id = "TEST_RECORD_9999"
        cls.test_doc = {
            "_id": cls.test_id,
            "name": "SeedDog",
            "animal_type": "Dog",
            "breed": "Labrador",
            "medical_notes": {"vaccinated": True, "neutered": True},
            "outcome_type": "Adpotion"
        }
    
    @classmethod    
    def tearDownClass(cls):
        """Clean up test records after all tests complete."""
        cls.shelter.collection.delete_many({"_id": cls.test_id})
        
    def setUp(self):
        """ Ensure clean state before each test."""
        self.shelter.collection.delete_one({"_id": self.test_id})
        
    def tearDown(self):
        """Ensure clean state after each test."""
        self.shelter.collection.delete_many({"_id": self.test_id})
            
    # ---------------------------CREATE TESTS----------------------------------
    def test_create_success(self):
        """Verify successful insertion of a valid document"""
        result = self.shelter.create(self.test_doc)
        self.assertTrue(result)
        # Verify persistence in database
        records = self.shelter.read({"_id": self.test_id})
        self.assertEqual(len(records), 1)
        
    def test_create_nested_subdocuments(self):
        """Verify successful insertion of valid document with an embedded dictionary"""
        nested_doc = {
            "_id": "NESTED_DOC_001",
            "name": "NestedSubdocDog",
            "animal_type": "Dog",
            "breed": "Pug",
            "outcome_type": "Adoption",
            "address": {
                "street": "123 Bark Lane",
                "city": "Raleigh",
                "state": "NC"
            }
        }
        # Clean potential collision, insert, and assert
        self.shelter.collection.delete_many({"_id": "NESTED_DOC_001"})
        result = self.shelter.create(nested_doc)
        self.assertTrue(result)
        # Verify persistence in database
        records = self.shelter.read({"name": "NestedSubdocDog"})
        self.assertEqual(len(records), 1)
        self.assertEqual(records[0]["address"]["city"], "Raleigh")
        self.shelter.collection.delete_many({"_id": "NESTED_DOC_001"})
    
    def test_create_duplicate_key(self):
        """Verify Duplicate _id returns False and does not crash runtime."""
        self.shelter.create(self.test_doc) # create initial instance of key
        duplicate_result = self.shelter.create(self.test_doc) # attempt to create duplicate
        self.assertFalse(duplicate_result) # Assert that the operation reported failure
        
        # Verify original document remained untouched
        existing_records = self.shelter.read({"_id": self.test_id})
        self.assertEqual(len(existing_records), 1)
        self.assertEqual(existing_records[0].get("name"), "SeedDog")
        
    def test_create_invalid_input(self):
        """Verify non-dict, empty dict, string, and list input return false"""
        self.assertFalse(self.shelter.create(None))
        self.assertFalse(self.shelter.create({}))
        self.assertFalse(self.shelter.create("string_not_dict"))
        self.assertFalse(self.shelter.create(["list", "not_dict"]))
                
    # ---------------------------READ TESTS-----------------------------------
    def test_read_existing_record(self):
        """Verify querying an existing document returns a non-empty list"""
        self.shelter.create(self.test_doc)
        records = self.shelter.read({"_id": self.test_id})
        self.assertIsInstance(records, list)
        self.assertGreater(len(records), 0)
        self.assertEqual(records[0]["name"], "SeedDog")
             
    def test_read_empty_query_criteria(self):
        """Verify querying with {} performs a wildcard search returning records."""
        records = self.shelter.read({})
        self.assertIsInstance(records, list)
        self.assertGreater(len(records), 0)
    
    def test_read_nonexistent_records(self):
        """Verify query for non-existent records safely returns an empty list."""
        records = self.shelter.read({"_id": "NONEXISTENT_999999"})
        self.assertEqual(records, [])  
         
    def test_read_multiple_matches(self): 
        """Verify querying common fields return multiple cursor records."""
        records = self.shelter.read({"animal_type": "Dog"})
        self.assertIsInstance(records, list)
        self.assertGreater(len(records), 1)

    def test_read_invalid_input_types(self):
        """Verify non-dict query input safely returns []."""
        self.assertEqual(self.shelter.read(None), [])
        self.assertEqual(self.shelter.read("not_a_dict"), [])
        self.assertEqual(self.shelter.read(["not", "a", "dict"]), [])
        self.assertEqual(self.shelter.read([1, 2, 3]), [])
        
    # ---------------------------UPDATE TESTS---------------------------------
    def test_update_success_with_raw_fields(self):
        """Verify updating a raw dict auto-wrapps in $set"""
        self.shelter.create(self.test_doc)
        modified_count = self.shelter.update(
            {"_id": self.test_id},
            {"outcome_type": "Transfer"}
        )
        self.assertEqual(modified_count, 1)
                         
        updated_records = self.shelter.read({"_id": self.test_id})
        self.assertEqual(updated_records[0]["outcome_type"], "Transfer")
        
    def test_update_success_with_atomic_operator(self):
        """Verify updating when caller explicitly passes MongoDB $set syntax"""
        self.shelter.create(self.test_doc)
        modified_count = self.shelter.update(
            {"_id": self.test_id},
            {"$set": {"breed": "mixed"}}
        )
        self.assertEqual(modified_count, 1)
                         
        updated_records = self.shelter.read({"_id": self.test_id})
        self.assertEqual(updated_records[0]["breed"], "mixed")
    
    def test_update_invalid_or_empty_inputs(self):
        """Verify non-dict or empty updates return 0 modified count."""
        self.shelter.create(self.test_doc)
        self.assertEqual(self.shelter.update(None, {"name": "Test"}), 0)
        self.assertEqual(self.shelter.update({"_id": self.test_id}, {}), 0)
        self.assertEqual(self.shelter.update({"_id": self.test_id}, None), 0)                

    # ---------------------------DELETE TESTS---------------------------------
    def test_delete_success(self):
        """Verify successful deletion of target document"""
        self.shelter.create(self.test_doc)
        deleted_count = self.shelter.delete({"_id": self.test_id})
        self.assertEqual(deleted_count, 1)

        # Verify document was deleted
        records = self.shelter.read({"_id": self.test_id})
        self.assertEqual(len(records), 0)
    
    def test_delete_invalid_or_empty_query(self):
        """Verify guard clause prevents ocllection wipe on empty/null query."""
        self.assertEqual(self.shelter.delete(None), 0)
        self.assertEqual(self.shelter.delete({}), 0)
        self.assertEqual(self.shelter.delete("invalid_type"), 0)  
        self.assertEqual(self.shelter.delete(["not", "a", "dict"]), 0)
                         
 
# Run Test Suite in Jupyter Notebook
unittest.main(argv=[''], verbosity=2, exit=False)

test_create_duplicate_key (__main__.TestAnimalShelterModule.test_create_duplicate_key)
Verify Duplicate _id returns False and does not crash runtime. ... ok
test_create_invalid_input (__main__.TestAnimalShelterModule.test_create_invalid_input)
Verify non-dict, empty dict, string, and list input return false ... ok
test_create_nested_subdocuments (__main__.TestAnimalShelterModule.test_create_nested_subdocuments)
Verify successful insertion of valid document with an embedded dictionary ... ok
test_create_success (__main__.TestAnimalShelterModule.test_create_success)
Verify successful insertion of a valid document ... ok
test_delete_invalid_or_empty_query (__main__.TestAnimalShelterModule.test_delete_invalid_or_empty_query)
Verify guard clause prevents ocllection wipe on empty/null query. ... ok
test_delete_success (__main__.TestAnimalShelterModule.test_delete_success)
Verify successful deletion of target document ... ok
test_read_empty_query_criteria (__main__.TestAnimalShelterModule.tes

An error occurred during [C]RUD- CREATE: E11000 duplicate key error collection: aac.animals index: _id_ dup key: { _id: "TEST_RECORD_9999" }, full error: {'index': 0, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: aac.animals index: _id_ dup key: { _id: "TEST_RECORD_9999" }', 'keyPattern': {'_id': 1}, 'keyValue': {'_id': 'TEST_RECORD_9999'}}


test_read_multiple_matches (__main__.TestAnimalShelterModule.test_read_multiple_matches)
Verify querying common fields return multiple cursor records. ... ok
test_read_nonexistent_records (__main__.TestAnimalShelterModule.test_read_nonexistent_records)
Verify query for non-existent records safely returns an empty list. ... ok
test_update_invalid_or_empty_inputs (__main__.TestAnimalShelterModule.test_update_invalid_or_empty_inputs)
Verify non-dict or empty updates return 0 modified count. ... ok
test_update_success_with_atomic_operator (__main__.TestAnimalShelterModule.test_update_success_with_atomic_operator)
Verify updating when caller explicitly passes MongoDB $set syntax ... ok
test_update_success_with_raw_fields (__main__.TestAnimalShelterModule.test_update_success_with_raw_fields)
Verify updating a raw dict auto-wrapps in $set ... ok

----------------------------------------------------------------------
Ran 14 tests in 0.345s

OK
